# Notebook 2: Create Pinecone Vector Database

This notebook creates a Pinecone vector database from your processed chunks.

**Prerequisites:**
- Completed Notebook 1 (chunks file created)
- Pinecone API key in .env file
- OpenAI API key in .env file (for embeddings)

## 1. Setup & Imports

In [1]:
# Install required packages (run once)
!pip install pinecone-client python-dotenv openai langchain langchain-openai langchain-pinecone -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
import time

# Load environment variables
load_dotenv()

print("✓ Imports successful")

✓ Imports successful


## 2. Load API Keys

In [5]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify keys are loaded
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found in .env file")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("✓ API keys loaded successfully")

✓ API keys loaded successfully


## 3. Load Your Processed Chunks

In [7]:
# Load chunks from your processed data
chunks_file = '../data/processed/chunks_aQf6Q8t1FQE.json'  # Adjust path if needed

with open(chunks_file, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"✓ Loaded {len(chunks)} chunks")
print(f"\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

✓ Loaded 10 chunks

Sample chunk:
{
  "chunk_id": "aQf6Q8t1FQE_chunk_0",
  "video_id": "aQf6Q8t1FQE",
  "video_title": "An Introduction to Stress and Strain",
  "channel": "The Efficient Engineer",
  "chunk_index": 0,
  "total_chunks": 10,
  "text": "Stress and strain are fundamental concepts\nthat are used to describe how a body responds to external loads. In this video we'll explore these concepts\nusing the simple example of a loaded bar. Here we have a solid metal bar that is loaded\nby two equal but opposite forces. We refer to this as uniaxial loading, because\nall of the applied loads are acting along the same axis. The two forces are pulling the bar, causing\nit to stretch. Internal forces will develop within the bar\nto resist these applied forces. We can expose these internal forces by making\nan imaginary cut through the bar. I chose to remove the right side of the bar,\nbut I could have removed the left side instead. For any imaginary cut like this one, the internal\nforces

## 4. Initialize Pinecone

In [8]:
# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

print("✓ Pinecone client initialized")

# List existing indexes
existing_indexes = pc.list_indexes()
print(f"\nExisting indexes: {[index.name for index in existing_indexes]}")

✓ Pinecone client initialized

Existing indexes: []


## 5. Create Pinecone Index

**Important Settings:**
- **Dimension**: 1536 (for OpenAI text-embedding-ada-002)
- **Metric**: cosine (recommended for text)
- **Cloud**: aws (free tier)
- **Region**: us-east-1 (free tier)

In [9]:
# Configuration
INDEX_NAME = 'efficient-engineer'  # You can change this name
DIMENSION = 1536  # OpenAI embeddings dimension
METRIC = 'cosine'

# Check if index already exists
index_names = [index.name for index in pc.list_indexes()]

if INDEX_NAME not in index_names:
    print(f"Creating new index: {INDEX_NAME}")
    
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'  # Free tier region
        )
    )
    
    # Wait for index to be ready
    print("Waiting for index to be ready...")
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(1)
    
    print(f"✓ Index '{INDEX_NAME}' created successfully")
else:
    print(f"✓ Index '{INDEX_NAME}' already exists")

# Connect to index
index = pc.Index(INDEX_NAME)

# Get index stats
stats = index.describe_index_stats()
print(f"\nIndex stats:")
print(f"  Total vectors: {stats['total_vector_count']}")
print(f"  Dimension: {stats['dimension']}")

Creating new index: efficient-engineer
Waiting for index to be ready...
✓ Index 'efficient-engineer' created successfully

Index stats:
  Total vectors: 0
  Dimension: 1536


## 6. Initialize OpenAI Embeddings

In [10]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,
    model="text-embedding-ada-002"  # Standard model, 1536 dimensions
)

print("✓ OpenAI embeddings initialized")

# Test embedding
test_text = "This is a test"
test_embedding = embeddings.embed_query(test_text)
print(f"\nTest embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

✓ OpenAI embeddings initialized

Test embedding dimension: 1536
First 5 values: [-0.008058104664087296, -0.0036456480156630278, -0.0003476669080555439, -0.0056320796720683575, -0.024551108479499817]


## 7. Prepare Documents for Pinecone

In [11]:
# Convert chunks to LangChain Document format
documents = []

for chunk in chunks:
    # Create metadata (Pinecone supports various types)
    metadata = {
        'chunk_id': chunk['chunk_id'],
        'video_id': chunk['video_id'],
        'video_title': chunk['video_title'],
        'channel': chunk['channel'],
        'chunk_index': chunk['chunk_index'],
        'video_url': chunk['video_url'],
    }
    
    # Add optional metadata fields if they exist
    if 'metadata' in chunk:
        if 'upload_date' in chunk['metadata']:
            metadata['upload_date'] = chunk['metadata']['upload_date']
        if 'tags' in chunk['metadata']:
            # Convert tags list to string for Pinecone
            metadata['tags'] = ', '.join(chunk['metadata']['tags'][:5])  # Limit to 5 tags
    
    # Create Document object
    doc = Document(
        page_content=chunk['text'],
        metadata=metadata
    )
    documents.append(doc)

print(f"✓ Prepared {len(documents)} documents")
print(f"\nSample document:")
print(f"Content: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

✓ Prepared 10 documents

Sample document:
Content: Stress and strain are fundamental concepts
that are used to describe how a body responds to external...
Metadata: {'chunk_id': 'aQf6Q8t1FQE_chunk_0', 'video_id': 'aQf6Q8t1FQE', 'video_title': 'An Introduction to Stress and Strain', 'channel': 'The Efficient Engineer', 'chunk_index': 0, 'video_url': 'https://www.youtube.com/watch?v=aQf6Q8t1FQE', 'upload_date': '20200211', 'tags': 'stress, strain, shear stress, shear strain, stress element'}


## 8. Upload Documents to Pinecone

**Note:** This will create embeddings and upload to Pinecone. May take a few minutes depending on number of chunks.

In [12]:
print(f"Uploading {len(documents)} documents to Pinecone...")
print("This may take a few minutes...\n")

# Create vector store and upload documents
vectorstore = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    index_name=INDEX_NAME
)

print("✓ Documents uploaded successfully!")

# Verify upload
time.sleep(2)  # Wait for indexing
stats = index.describe_index_stats()
print(f"\nFinal index stats:")
print(f"  Total vectors: {stats['total_vector_count']}")
print(f"  Expected: {len(documents)}")

Uploading 10 documents to Pinecone...
This may take a few minutes...

✓ Documents uploaded successfully!

Final index stats:
  Total vectors: 10
  Expected: 10


## 9. Test Vector Search

In [13]:
# Test search with a sample query
test_query = "What is stress in engineering?"

print(f"Test query: '{test_query}'\n")
print("-" * 80)

# Search for similar documents
results = vectorstore.similarity_search(
    query=test_query,
    k=3  # Return top 3 results
)

# Display results
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"Video: {doc.metadata['video_title']}")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"URL: {doc.metadata['video_url']}")
    print("-" * 80)

Test query: 'What is stress in engineering?'

--------------------------------------------------------------------------------

Result 1:
Video: An Introduction to Stress and Strain
Text: the forces are stretching the bar. If the forces were trying to shorten the bar,
we would have a compressive stress. The sign convention that is normally used
is that tensile stresses are positive val...
URL: https://www.youtube.com/watch?v=aQf6Q8t1FQE
--------------------------------------------------------------------------------

Result 2:
Video: An Introduction to Stress and Strain
Text: Stress and strain are fundamental concepts
that are used to describe how a body responds to external loads. In this video we'll explore these concepts
using the simple example of a loaded bar. Here we...
URL: https://www.youtube.com/watch?v=aQf6Q8t1FQE
--------------------------------------------------------------------------------

Result 3:
Video: An Introduction to Stress and Strain
Text: acting on the cross-se

## 10. Test Search with Scores

In [14]:
# Search with similarity scores
test_query = "Explain buckling in structures"

print(f"Test query: '{test_query}'\n")
print("-" * 80)

results_with_scores = vectorstore.similarity_search_with_score(
    query=test_query,
    k=3
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\nResult {i} (Score: {score:.4f}):")
    print(f"Video: {doc.metadata['video_title']}")
    print(f"Text: {doc.page_content[:200]}...")
    print("-" * 80)

Test query: 'Explain buckling in structures'

--------------------------------------------------------------------------------

Result 1 (Score: 0.7787):
Video: An Introduction to Stress and Strain
Text: Stress and strain are fundamental concepts
that are used to describe how a body responds to external loads. In this video we'll explore these concepts
using the simple example of a loaded bar. Here we...
--------------------------------------------------------------------------------

Result 2 (Score: 0.7727):
Video: An Introduction to Stress and Strain
Text: like this one show that there is an initial region for low strain values where the relationship
between stress and strain is linear. Deformations occurring in this region are
fully reversed when the l...
--------------------------------------------------------------------------------

Result 3 (Score: 0.7785):
Video: An Introduction to Stress and Strain
Text: looks like this. That's it for this introduction to stress
and strain. H

## 11. Create Retriever for RAG

This retriever can be used directly in your LangChain RAG agent.

In [16]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 chunks
)

print("✓ Retriever created")

# Test retriever
test_query = "What is the finite element method?"
retrieved_docs = retriever.invoke(test_query)

print(f"\nTest retrieval for: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.metadata['video_title']}")
    print(f"   {doc.page_content[:100]}...\n")

✓ Retriever created

Test retrieval for: 'What is the finite element method?'
Retrieved 3 documents

1. An Introduction to Stress and Strain
   modulus. Although I have discussed normal and shear
stresses separately so far, the stress state at ...

2. An Introduction to Stress and Strain
   looking at the stresses acting on a small element within our bar. We have a shear stress on one face...

3. An Introduction to Stress and Strain
   acting on the cross-section created by our cut will be equal to the effect of the applied
external f...



## 12. Save Configuration

Save your Pinecone configuration for easy loading in other scripts.

In [17]:
# Save configuration
config = {
    'index_name': INDEX_NAME,
    'dimension': DIMENSION,
    'metric': METRIC,
    'total_chunks': len(documents),
    'embedding_model': 'text-embedding-ada-002',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

config_file = '../data/pinecone_config.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Configuration saved to: {config_file}")
print(f"\nConfiguration:")
print(json.dumps(config, indent=2))

✓ Configuration saved to: ../data/pinecone_config.json

Configuration:
{
  "index_name": "efficient-engineer",
  "dimension": 1536,
  "metric": "cosine",
  "total_chunks": 10,
  "embedding_model": "text-embedding-ada-002",
  "created_at": "2026-03-05 15:33:57"
}


## 13. Summary & Next Steps

In [18]:
print("="*80)
print("PINECONE VECTOR DATABASE SETUP COMPLETE!")
print("="*80)
print(f"\n✓ Index name: {INDEX_NAME}")
print(f"✓ Total vectors: {stats['total_vector_count']}")
print(f"✓ Dimension: {DIMENSION}")
print(f"✓ Metric: {METRIC}")
print(f"\n✓ Vector store ready for RAG!")
print(f"\nNext steps:")
print("  1. Build your RAG agent (Notebook 3)")
print("  2. Create chat interface")
print("  3. Test with engineering questions")
print("\n" + "="*80)

PINECONE VECTOR DATABASE SETUP COMPLETE!

✓ Index name: efficient-engineer
✓ Total vectors: 10
✓ Dimension: 1536
✓ Metric: cosine

✓ Vector store ready for RAG!

Next steps:
  1. Build your RAG agent (Notebook 3)
  2. Create chat interface
  3. Test with engineering questions



## 14. Helper: Load Existing Pinecone Index

Use this cell to reconnect to your index in future sessions.

In [ ]:
# Run this cell to reconnect to existing index
def load_pinecone_vectorstore(index_name='efficient-engineer'):
    """
    Load existing Pinecone vector store
    """
    from pinecone import Pinecone
    from langchain_openai import OpenAIEmbeddings
    from langchain_pinecone import PineconeVectorStore
    import os
    from dotenv import load_dotenv
    
    load_dotenv()
    
    # Initialize
    pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
    embeddings = OpenAIEmbeddings(openai_api_key=os.getenv('OPENAI_API_KEY'))
    
    # Connect to existing index
    vectorstore = PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings
    )
    
    print(f"✓ Connected to Pinecone index: {index_name}")
    return vectorstore

# Example usage:
# vectorstore = load_pinecone_vectorstore('efficient-engineer')
# retriever = vectorstore.as_retriever()

## 15. Optional: View All Metadata Fields

In [ ]:
# Check what metadata is stored
sample_results = vectorstore.similarity_search("engineering", k=1)

if sample_results:
    print("Metadata fields in Pinecone:")
    print("-" * 40)
    for key, value in sample_results[0].metadata.items():
        print(f"  {key}: {value}")

## 16. Optional: Delete Index (if needed)

⚠️ **Warning:** This will delete all your vectors!

In [ ]:
# Uncomment to delete index
# pc.delete_index(INDEX_NAME)
# print(f"✗ Index '{INDEX_NAME}' deleted")